--- To be used to ingest new files into the database

In [ ]:
%pip install "psycopg[binary]" psycopg2-binary asyncpg python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 6.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 5.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 636.9/636.9 kB 5.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [asyncpg]m2/4 [psycopg]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
import requests
from datetime import datetime, timedelta, timezone
import psycopg
from dotenv import load_dotenv
import os

In [28]:
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

In [29]:
# ---- CONFIG ----
measure_id = "0009-level-tidal_level-i-15_min-mAOD"

# ---- FETCH DATA FROM EA API ----

url = f"https://environment.data.gov.uk/flood-monitoring/id/measures/{measure_id}/readings"

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:

        cur.execute("""
        CREATE TABLE IF NOT EXISTS richmond_tidal_levels (
            ts TIMESTAMPTZ PRIMARY KEY,
            water_level DOUBLE PRECISION NOT NULL,
            station_id TEXT NOT NULL,
            created_at TIMESTAMPTZ DEFAULT NOW()
        );
        """)

        # Get last timestamp
        cur.execute("SELECT MAX(ts) FROM richmond_tidal_levels;")
        last_ts = cur.fetchone()[0]

# ---- FETCH DATA ----

params = {"_limit": 10000}

if last_ts:
    params["since"] = last_ts.isoformat()

r = requests.get(url, params=params)
r.raise_for_status()

data = r.json().get("items", [])

# ---- INSERT NEW DATA ----

with psycopg.connect(DATABASE_URL) as conn:
    with conn.cursor() as cur:

        for reading in data:
            cur.execute("""
                INSERT INTO richmond_tidal_levels (ts, water_level, station_id)
                VALUES (%s, %s, %s)
                ON CONFLICT (ts) DO NOTHING;
            """, (
                reading["dateTime"],
                reading["value"],
                "0009"
            ))

    conn.commit()

print(f"Inserted {len(data)} new readings.")

Inserted 0 new readings.
